In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import train_test_split_edges, negative_sampling
from torch_geometric.transforms import NormalizeFeatures
from sklearn.metrics import roc_auc_score, average_precision_score

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Koristimo:", device)

dataset = Planetoid(root='./data', name='Cora', transform=NormalizeFeatures())
data = dataset[0]
data = train_test_split_edges(data)
data = data.to(device)

Koristimo: cpu


/tmp/ipykernel_50039/1868854233.py:18: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  data = train_test_split_edges(data)


In [22]:
from torch_geometric.nn import GCNConv

class VGAE(nn.Module):
    def __init__(self, in_channels, hid_channels, out_channels):
        super().__init__()
        # Zajednički prvi sloj — isti kao u GAE
        self.conv1 = GCNConv(in_channels, hid_channels)
        
        # RAZLIKA: umesto jednog drugog sloja, imamo DVA izlazna sloja
        self.conv_mu      = GCNConv(hid_channels, out_channels)  # uči srednje vrednosti
        self.conv_logstd  = GCNConv(hid_channels, out_channels)  # uči log std devijacije

    def encode(self, x, edge_index):
        # Zajednički deo — identičan GAE-u
        h = self.conv1(x, edge_index).relu()   # [2708, 64]
        
        # RAZLIKA: dva izlaza umesto jednog
        self.mu     = self.conv_mu(h, edge_index)      # [2708, 32] — srednje vrednosti
        self.logstd = self.conv_logstd(h, edge_index)  # [2708, 32] — log std devijacije
        self.logstd = self.logstd.clamp(max=10)        # numerička stabilnost — sprečava eksploziju
        
        # Uzorkovanje: z = mu + epsilon * exp(logstd)
        # exp(logstd) = sigma (prava std devijacija, uvek pozitivna)
        if self.training:
            # Tokom treninga uzorkujemo — model uči da cela raspodela bude dobra
            epsilon = torch.randn_like(self.mu)        # [2708, 32] ~ N(0, I)
            z = self.mu + epsilon * self.logstd.exp()
        else:
            # Tokom evaluacije koristimo samo mu — deterministički, stabilniji
            z = self.mu
        
        return z
        
    def kl_loss(self):
        # KL divergencija između naučene raspodele N(mu, sigma^2) i N(0, I)
        # Formula: -0.5 * mean(1 + 2*logstd - mu^2 - exp(2*logstd))
        # Ovo je analitičko rešenje KL divergencije između dve Gaussove raspodele
        return -0.5 * torch.mean(
            1 + 2 * self.logstd - self.mu.pow(2) - (2 * self.logstd).exp()
        )

    # Decode i decode_all su IDENTIČNI GAE-u
    def decode(self, z, edge_label_index):
        return (z[edge_label_index[0]] * z[edge_label_index[1]]).sum(dim=-1)

    def decode_all(self, z):
        prob_adj = z @ z.t()
        prob_adj = prob_adj.sigmoid()
        return (prob_adj > 0.97).nonzero(as_tuple=False).t()

In [23]:
input_channels  = dataset.num_features  # 1433
hidden_channels = 64
output_channels = 32

model     = VGAE(in_channels=input_channels, hid_channels=hidden_channels, out_channels=output_channels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()

In [24]:
def train():
    model.train()
    optimizer.zero_grad()
    
    z = model.encode(data.x, data.train_pos_edge_index)
    
    neg_edge_index = negative_sampling(
        edge_index=data.train_pos_edge_index,
        num_nodes=data.num_nodes,
        num_neg_samples=data.train_pos_edge_index.shape[1]
    )
    
    edge_index = torch.cat([data.train_pos_edge_index, neg_edge_index], dim=1)
    
    labels = torch.cat([
        torch.ones(data.train_pos_edge_index.shape[1]),
        torch.zeros(neg_edge_index.shape[1])
    ]).to(device)
    
    out  = model.decode(z, edge_index)
    
    # JEDINA RAZLIKA u train() funkciji:
    bce_loss = criterion(out, labels)
    kl_loss  = model.kl_loss()
    loss     = bce_loss + 0.001 * kl_loss   # ukupan loss = rekonstrukcija + regularizacija
    
    loss.backward()
    optimizer.step()
    
    return loss.item()

In [26]:
@torch.no_grad()
def evaluate(pos_edge_index, neg_edge_index):
    model.eval()  # ovo triggeruje z = mu umesto uzorkovanja u encode()
    z = model.encode(data.x, data.train_pos_edge_index)
    
    edge_index = torch.cat([pos_edge_index, neg_edge_index], dim=1)
    out        = model.decode(z, edge_index).sigmoid().cpu().numpy()
    
    labels = torch.cat([
        torch.ones(pos_edge_index.shape[1]),
        torch.zeros(neg_edge_index.shape[1])
    ]).numpy()
    
    auc = roc_auc_score(labels, out)
    ap  = average_precision_score(labels, out)
    return auc, ap

# Ovo je identično GAE-u u potpunosti
val_neg  = negative_sampling(data.val_pos_edge_index,  data.num_nodes, data.val_pos_edge_index.shape[1])
test_neg = negative_sampling(data.test_pos_edge_index, data.num_nodes, data.test_pos_edge_index.shape[1])

best_val_auc = 0
for epoch in range(1, 301):
    loss = train()
    
    if epoch % 10 == 0:
        val_auc, val_ap = evaluate(data.val_pos_edge_index, val_neg)
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Val AUC: {val_auc:.4f} | Val AP: {val_ap:.4f}")
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), 'best_vgae_model.pt')

model.load_state_dict(torch.load('best_vgae_model.pt'))
test_auc, test_ap = evaluate(data.test_pos_edge_index, test_neg)
print(f"\n=== FINALNI REZULTATI (VGAE) ===")
print(f"Test AUC: {test_auc:.4f}")
print(f"Test AP:  {test_ap:.4f}")

Epoch 010 | Loss: 0.4548 | Val AUC: 0.8867 | Val AP: 0.8820
Epoch 020 | Loss: 0.4521 | Val AUC: 0.8878 | Val AP: 0.8815
Epoch 030 | Loss: 0.4444 | Val AUC: 0.8927 | Val AP: 0.8867
Epoch 040 | Loss: 0.4478 | Val AUC: 0.8939 | Val AP: 0.8888
Epoch 050 | Loss: 0.4456 | Val AUC: 0.8978 | Val AP: 0.8930
Epoch 060 | Loss: 0.4441 | Val AUC: 0.9023 | Val AP: 0.8962
Epoch 070 | Loss: 0.4426 | Val AUC: 0.9046 | Val AP: 0.8991
Epoch 080 | Loss: 0.4410 | Val AUC: 0.9057 | Val AP: 0.9003
Epoch 090 | Loss: 0.4372 | Val AUC: 0.9070 | Val AP: 0.9008
Epoch 100 | Loss: 0.4392 | Val AUC: 0.9066 | Val AP: 0.9011
Epoch 110 | Loss: 0.4335 | Val AUC: 0.9067 | Val AP: 0.9034
Epoch 120 | Loss: 0.4324 | Val AUC: 0.9092 | Val AP: 0.9058
Epoch 130 | Loss: 0.4395 | Val AUC: 0.9114 | Val AP: 0.9071
Epoch 140 | Loss: 0.4255 | Val AUC: 0.9110 | Val AP: 0.9082
Epoch 150 | Loss: 0.4337 | Val AUC: 0.9103 | Val AP: 0.9083
Epoch 160 | Loss: 0.4265 | Val AUC: 0.9131 | Val AP: 0.9114
Epoch 170 | Loss: 0.4262 | Val AUC: 0.91